In [1]:
import numpy as np

class Node:

    def __init__(self, pos: tuple, g_cost: float, h_cost: float):
        self.pos = pos # (x, y) tuple
        self.g_cost = g_cost
        self.h_cost = h_cost
        self.f_cost = self.g_cost + self.h_cost
        self.parent = None  # previous node
    
    def __lt__(self, other):
        return self.f_cost < other.f_cost

class AStar:

    def __init__(self, map_grid):
        self.open = []  # open list
        self.closed = []  # closed list
        self.map_grid = map_grid
        
    def search(self, start_node, goal_node):

        self.open.append(start_node)

        while self.open:
            # sort open list to get node with lowest cost first
            self.open.sort()
            current_node = self.open.pop(0)

            # add current node to closed list
            self.closed.append(current_node)

            if current_node.pos == goal_node.pos:
                # reached goal node
                return self.reconstruct_path(start_node, goal_node)

            # check every neighbor
            neighbors = self.get_neighbors(current_node)

            for neighbor_node in neighbors:
                if neighbor_node.pos in [n.pos for n in self.closed]:
                    continue

                g_cost = current_node.g_cost + 1  # cost to move
                h_cost = self.heuristic(neighbor_node, goal_node)
                f_cost = g_cost + h_cost

                # check if we found a cheaper path
                if neighbor_node.pos in [n.pos for n in self.open]:
                    for open_node in self.open:
                        if open_node.pos == neighbor_node.pos and open_node.f_cost > f_cost:
                            self.update_node(open_node, g_cost, h_cost, current_node)
                else:
                    self.update_node(neighbor_node, g_cost, h_cost, current_node)
                    self.open.append(neighbor_node)

        # no path found
        return None

    def get_neighbors(self, node):
        dirs = [[1, 0], [0, 1], [-1, 0], [0, -1]]
        neighbors = []

        for dir in dirs:
            neighbor_pos = (node.pos[0] + dir[0], node.pos[1] + dir[1])

            # check if new pos is in bounds
            if (0 <= neighbor_pos[0] < self.map_grid.shape[0] and
                0 <= neighbor_pos[1] < self.map_grid.shape[1]):

                # check if traversable
                if self.map_grid[neighbor_pos] != 1:
                    # Create a new Node for the neighbor
                    neighbor_node = Node(neighbor_pos, 0, 0)  # g_cost and h_cost will be updated later
                    neighbors.append(neighbor_node)

        return neighbors

    def heuristic(self, node, goal):
        # Manhattan distance
        d = abs(node.pos[0] - goal.pos[0]) + abs(node.pos[1] - goal.pos[1])
        return d

    def reconstruct_path(self, start_node, goal_node):
        path = [goal_node]
        current = goal_node

        while current != start_node:
            if current.parent is None:
                return None  
            path.append(current.parent)
            current = current.parent

        return path[::-1]  # reverse path

    def update_node(self, node, g_cost, h_cost, parent_node):
        node.g_cost = g_cost
        node.h_cost = h_cost
        node.f_cost = g_cost + h_cost
        node.parent = parent_node




In [2]:
import numpy as np
import matplotlib.pyplot as plt

# Reusing Node and AStar classes

# Define the map grid (0 for open space, 1 for obstacles)
map_grid = np.array([
    [0, 0, 0, 0, 0],
    [1, 1, 0, 1, 0],
    [0, 0, 0, 1, 0],
    [0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0]
])

# Create start and goal nodes
start_node = Node((0, 0), 0, 0)
goal_node = Node((4, 4), 0, 0)

# Initialize A* algorithm
a_star = AStar(map_grid)
path = a_star.search(start_node, goal_node)

